# 3D reporter timelapse — 05b_reporter_spatial_context

**Feeds:** Fig 5h, ED Fig 10d

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Reporter Cooperativity Exploration

This notebook is now dedicated to **Theme 1** of the cooperativity exploration:

- does `YFP` usually appear within an existing `RFP` context?
- how sensitive is that picture to the `YFP` threshold?
- what threshold-independent views, if any, support the same picture?

The timing and alignment themes now live in separate notebooks:

- `05c_reporter_temporal_ordering`
- `05d_reporter_alignment_switchlike`


## Interpretation Frame

The goal here is narrow and spatial:

1. quantify how much of the `YFP`-positive domain lies inside the `RFP`-positive domain
2. use that containment summary to help judge a sensible `YFP` threshold while holding `RFP` fixed
3. inspect representative examples and outliers directly
4. add a small number of threshold-independent supporting views, but only if they are easy to read

This notebook is not trying to resolve timing or mechanism by itself. It is meant to settle the domain-definition and spatial-context questions before those later analyses.


## Upstream Signal Preprocessing

The thresholded reporter metrics shown here are inherited from notebook `05`; this notebook does **not** reprocess images from scratch.

The upstream preprocessing chain is:

1. apply reporter-specific illumination correction to the raw `FOXF1-RFP` and `BMP4-YFP` images using the shared channel-wide illumination fields estimated earlier in the pipeline
2. for each image/frame separately, measure the whole off-cyst pixel median **after** illumination correction and subtract that per-image scalar background
3. after that subtraction, estimate the reporter-negative within-cyst baseline and sigma from early corrected cyst pixels, then call positive pixels at the chosen `N sigma` threshold

So the thresholded areas, positive fractions, and positive-region intensity summaries in these notebooks all sit on top of:

- illumination correction
- per-image off-cyst median background subtraction
- within-cyst baseline / sigma estimation for thresholding

By default in the manuscript-facing reporter notebooks, the thresholded positive-fraction comparisons use:

- `FOXF1-RFP`: `4 sigma`
- `BMP4-YFP`: `3 sigma`


## Setup


In [ ]:
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import Image, Markdown, display
from scipy.ndimage import gaussian_filter
from scipy import stats
from skimage import exposure, morphology
from skimage.measure import find_contours
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

qc_dir = ROOT / "scripts" / "qc"
if str(qc_dir) not in sys.path:
    sys.path.insert(0, str(qc_dir))

from notebook_display_helpers import (
    TIME_DISPLAY_OFFSET_HOURS,
    display_time_hours as _display_time_hours,
    format_display_hours as _format_display_hours,
    format_display_hour_range as _format_display_hour_range,
    offset_time_hours_df as _offset_time_hours_df,
    set_display_time_axis as _set_display_time_axis,
)
from notebook_figure_helpers import (
    NotebookFigureExportPaths,
    install_png_vector_savefig_exports,
)
from notebook_invariant_helpers import (
    THEME_DEFAULT_SIGMA_BY_REPORTER,
    assert_no_excluded_keys_in_analysis,
    assert_theme_default_sigmas,
)
from reporter_cooperativity_shared import (
    REPORTER_COLORS,
    find_project_root,
)

ROOT = find_project_root(ROOT)
install_png_vector_savefig_exports()
FRAME_METRICS_PATH = ROOT / "results/tables/05_reporter_metrics_by_frame.tsv"
GLOBAL_THRESHOLDS_PATH = ROOT / "results/tables/05_global_reporter_thresholds.tsv"
BACKGROUND_OUTLIERS_PATH = ROOT / "results/tables/04b_background_stability_outliers.tsv"
MASK_DIR = ROOT / "results/ilastik/organoid_masks/full_dataset_v1"
CORRECTED_STACK_DIR = ROOT / "data/corrected_three_channel_position_stacks"

FIGURE_EXPORTS = NotebookFigureExportPaths.create(
    ROOT,
    "05b",
    alternate_strip_prefixes=("05b_",),
    candidate_strip_prefixes=("05b_",),
)
FIGURE_DIR = FIGURE_EXPORTS.figure_dir
ALTERNATE_FIGURE_DIR = FIGURE_EXPORTS.alternate_dir
CANDIDATE_FIGURE_DIR = FIGURE_EXPORTS.candidate_dir
TABLE_DIR = ROOT / "results/tables"

frame_metrics = pd.read_csv(FRAME_METRICS_PATH, sep="\t", low_memory=False)
global_thresholds = pd.read_csv(GLOBAL_THRESHOLDS_PATH, sep="\t")
background_outliers = pd.read_csv(BACKGROUND_OUTLIERS_PATH, sep="\t")
final_time_hours_by_position = frame_metrics.groupby("position_label")["time_hours"].max().to_dict()

threshold_lookup = {
    row["reporter"]: float(row["threshold_value"])
    for _, row in global_thresholds.iterrows()
}
threshold_params = {
    row["reporter"]: {
        "baseline_location": float(row["baseline_location"]),
        "baseline_scale": float(row["baseline_scale"]),
    }
    for _, row in global_thresholds.iterrows()
}
background_flagged_positions = set(background_outliers["position_label"].dropna().unique())
POSITION_REVIEW_SET = ["Pos11", "Pos17", "Pos19", "Pos20", "Pos24", "Pos26", "Pos40"]
DEFAULT_RFP_SIGMA = int(THEME_DEFAULT_SIGMA_BY_REPORTER["RFP"])
DEFAULT_YFP_SIGMA = int(THEME_DEFAULT_SIGMA_BY_REPORTER["YFP"])
GAP_BREAK_THRESHOLD_HOURS = 0.30
DISPLAY_REPORTER_LABELS = {"RFP": "FOXF1-RFP", "YFP": "BMP4-YFP"}
analysis_frame_keys = (
    frame_metrics.loc[~frame_metrics["exclude_from_analysis"].fillna(False), ["position_label", "time_index"]]
    .drop_duplicates()
    .sort_values(["position_label", "time_index"])
    .reset_index(drop=True)
)

assert_theme_default_sigmas(
    {"RFP": DEFAULT_RFP_SIGMA, "YFP": DEFAULT_YFP_SIGMA},
    context="05b Theme 1 default thresholds",
)
assert_no_excluded_keys_in_analysis(
    frame_metrics,
    analysis_frame_keys,
    key_columns=("position_label", "time_index"),
    context="05b usable image-frame keys",
)

default_rfp_threshold_value = threshold_params["RFP"]["baseline_location"] + DEFAULT_RFP_SIGMA * threshold_params["RFP"]["baseline_scale"]
default_yfp_threshold_value = threshold_params["YFP"]["baseline_location"] + DEFAULT_YFP_SIGMA * threshold_params["YFP"]["baseline_scale"]

display(
    Markdown(
        f'''
        **Loaded Theme 1 inputs**

        - frame-metric rows: `{len(frame_metrics):,}`
        - positions in frame table: `{frame_metrics["position_label"].nunique():,}`
        - background-QC-flagged positions excluded upstream: `{len(background_flagged_positions):,}`
        - Theme 1 default thresholds:
          - `FOXF1-RFP 4σ`: `{default_rfp_threshold_value:.1f}`
          - `BMP4-YFP 3σ`: `{default_yfp_threshold_value:.1f}`
        '''
    )
)


In [ ]:
def positive_mask_from_corrected(
    corrected: np.ndarray,
    organoid_mask: np.ndarray,
    threshold_value: float,
    closing_radius: int = 1,
    min_object_size: int = 16,
    hole_area: int = 16,
) -> np.ndarray:
    positive = organoid_mask & (corrected > float(threshold_value))
    if closing_radius > 0:
        positive = morphology.binary_closing(positive, morphology.disk(closing_radius))
    if min_object_size > 1:
        positive = morphology.remove_small_objects(positive, min_size=min_object_size)
    if hole_area > 1:
        positive = morphology.remove_small_holes(positive, area_threshold=hole_area)
    return positive & organoid_mask


def threshold_value_for_sigma(reporter: str, sigma_threshold: float) -> float:
    params = threshold_params[reporter]
    return float(params["baseline_location"] + float(sigma_threshold) * params["baseline_scale"])


def reporter_display(reporter: str) -> str:
    return DISPLAY_REPORTER_LABELS.get(str(reporter), str(reporter))


def display_time_hours(values):
    return _display_time_hours(values, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours(value: float, decimals: int = 1) -> str:
    return _format_display_hours(value, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hour_range(start: float, end: float, decimals: int = 0) -> str:
    return _format_display_hour_range(start, end, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def set_display_time_axis(ax, axis: str = "x", crowded: bool = False) -> None:
    _set_display_time_axis(ax, axis=axis, crowded=crowded, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    return _offset_time_hours_df(
        df,
        should_offset_column=lambda column: str(column).endswith("time_hours"),
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def figure_path(filename: str, candidate: bool = False) -> Path:
    return FIGURE_EXPORTS.figure_path(filename, candidate=candidate)


def organoid_mask_path(position_label: str, time_index: int) -> Path:
    position_index = int(position_label.replace("Pos", ""))
    return MASK_DIR / position_label / f"img_channel000_position{position_index:03d}_time{int(time_index):09d}_z000_mask.tiff"


corrected_stack_cache: dict[str, np.ndarray] = {}


def load_corrected_stack(position_label: str) -> np.ndarray:
    if position_label not in corrected_stack_cache:
        corrected_stack_cache[position_label] = tiff.imread(
            CORRECTED_STACK_DIR / f"{position_label}_corrected_three_channel_stack.tif"
        )
    return corrected_stack_cache[position_label]


def containment_by_time(
    yfp_sigma_threshold: int = 4,
    rfp_sigma_threshold: int = 4,
    min_yfp_pixels: int = 50,
) -> pd.DataFrame:
    yfp_threshold_value = threshold_value_for_sigma("YFP", yfp_sigma_threshold)
    rfp_threshold_value = threshold_value_for_sigma("RFP", rfp_sigma_threshold)
    rows = []

    valid_frames = (
        frame_metrics.loc[~frame_metrics["exclude_from_analysis"], ["position_label", "time_index", "time_hours"]]
        .drop_duplicates()
        .sort_values(["position_label", "time_index"])
    )

    for position_label, position_df in valid_frames.groupby("position_label", sort=True):
        if position_label in background_flagged_positions:
            continue

        stack = load_corrected_stack(position_label)
        rfp_stack = stack[:, 1].astype(float)
        yfp_stack = stack[:, 2].astype(float)

        for row in position_df.itertuples(index=False):
            time_index = int(row.time_index)
            time_hours = float(row.time_hours)
            mask_path = organoid_mask_path(position_label, time_index)
            if not mask_path.exists():
                continue

            organoid_mask = tiff.imread(mask_path) > 0
            rfp_mask = positive_mask_from_corrected(rfp_stack[time_index], organoid_mask, rfp_threshold_value)
            yfp_mask = positive_mask_from_corrected(yfp_stack[time_index], organoid_mask, yfp_threshold_value)

            yfp_count = int(yfp_mask.sum())
            if yfp_count < int(min_yfp_pixels):
                continue

            rows.append(
                {
                    "position_label": position_label,
                    "time_index": time_index,
                    "time_hours": time_hours,
                    "rfp_sigma_threshold": int(rfp_sigma_threshold),
                    "yfp_sigma_threshold": int(yfp_sigma_threshold),
                    "rfp_positive_pixels": int(rfp_mask.sum()),
                    "yfp_positive_pixels": yfp_count,
                    "intersection_pixels": int((rfp_mask & yfp_mask).sum()),
                    "containment_fraction": float((rfp_mask & yfp_mask).sum() / yfp_count),
                }
            )

    return pd.DataFrame(rows)


def load_or_build_containment_cache(
    cache_path: Path,
    yfp_sigma_threshold: int = 4,
    rfp_sigma_threshold: int = 4,
    min_yfp_pixels: int = 50,
) -> tuple[pd.DataFrame, str]:
    required_cols = {
        "position_label",
        "time_index",
        "time_hours",
        "rfp_sigma_threshold",
        "yfp_sigma_threshold",
        "rfp_positive_pixels",
        "yfp_positive_pixels",
        "intersection_pixels",
        "containment_fraction",
    }
    if cache_path.exists():
        cached = pd.read_csv(cache_path, sep="\t")
        if required_cols.issubset(cached.columns):
            if (
                set(cached["rfp_sigma_threshold"].dropna().unique()) == {int(rfp_sigma_threshold)}
                and set(cached["yfp_sigma_threshold"].dropna().unique()) == {int(yfp_sigma_threshold)}
            ):
                filtered = cached.loc[cached["yfp_positive_pixels"] >= int(min_yfp_pixels)].copy()
                return filtered, "reused existing"

    built = containment_by_time(
        yfp_sigma_threshold=yfp_sigma_threshold,
        rfp_sigma_threshold=rfp_sigma_threshold,
        min_yfp_pixels=min_yfp_pixels,
    )
    built.to_csv(cache_path, sep="\t", index=False)
    return built, "created new"


def build_shifted_log_transform(values: np.ndarray, q_floor: float = 0.001) -> tuple[float, callable]:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        floor = -1.0
    else:
        floor = float(np.quantile(finite, q_floor))
    epsilon = 1.0

    def transform(x):
        arr = np.asarray(x, dtype=float)
        return np.log10(np.maximum(arr - floor + epsilon, 1e-9))

    return floor, transform


def threshold_independent_pixel_data(
    max_samples_per_frame: int = 160,
    sample_frame_stride: int = 4,
    seed: int = 7,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(int(seed))
    valid_frames = (
        frame_metrics.loc[~frame_metrics["exclude_from_analysis"], ["position_label", "time_index", "time_hours"]]
        .drop_duplicates()
        .sort_values(["position_label", "time_index"])
    )

    pixel_rows = []
    weighted_rows = []

    for position_label, position_df in valid_frames.groupby("position_label", sort=True):
        if position_label in background_flagged_positions:
            continue

        stack = load_corrected_stack(position_label)
        rfp_stack = stack[:, 1].astype(float)
        yfp_stack = stack[:, 2].astype(float)

        for row in position_df.itertuples(index=False):
            time_index = int(row.time_index)
            time_hours = float(row.time_hours)
            mask_path = organoid_mask_path(position_label, time_index)
            if not mask_path.exists():
                continue

            organoid_mask = tiff.imread(mask_path) > 0
            if int(organoid_mask.sum()) < 10:
                continue

            rfp_values = rfp_stack[time_index][organoid_mask].astype(float)
            yfp_values = yfp_stack[time_index][organoid_mask].astype(float)
            rfp_z = (rfp_values - threshold_params["RFP"]["baseline_location"]) / threshold_params["RFP"]["baseline_scale"]
            yfp_z = (yfp_values - threshold_params["YFP"]["baseline_location"]) / threshold_params["YFP"]["baseline_scale"]

            if rfp_z.size > 1:
                order = np.argsort(rfp_z, kind="mergesort")
                rfp_percentile = np.empty_like(rfp_z, dtype=float)
                rfp_percentile[order] = (np.arange(rfp_z.size, dtype=float) + 0.5) / rfp_z.size
                yfp_weights = np.clip(yfp_z, a_min=0.0, a_max=None)
                if float(np.sum(yfp_weights)) > 0:
                    weighted_percentile = float(np.average(rfp_percentile, weights=yfp_weights))
                else:
                    weighted_percentile = float("nan")
            else:
                weighted_percentile = float("nan")

            weighted_rows.append(
                {
                    "position_label": position_label,
                    "time_index": time_index,
                    "time_hours": time_hours,
                    "yfp_weighted_mean_rfp_percentile": weighted_percentile,
                    "mask_pixel_count": int(rfp_z.size),
                    "positive_yfp_weight_sum": float(np.clip(yfp_z, a_min=0.0, a_max=None).sum()),
                }
            )

            if sample_frame_stride > 1 and (time_index % int(sample_frame_stride) != 0):
                continue
            sample_count = min(int(max_samples_per_frame), int(rfp_z.size))
            if sample_count <= 0:
                continue
            sample_indices = rng.choice(rfp_z.size, size=sample_count, replace=False)
            for sample_idx in sample_indices:
                pixel_rows.append(
                    {
                        "position_label": position_label,
                        "time_index": time_index,
                        "time_hours": time_hours,
                        "rfp_raw": float(rfp_values[sample_idx]),
                        "yfp_raw": float(yfp_values[sample_idx]),
                        "rfp_z": float(rfp_z[sample_idx]),
                        "yfp_z": float(yfp_z[sample_idx]),
                    }
                )

    return pd.DataFrame(pixel_rows), pd.DataFrame(weighted_rows)


def load_or_build_threshold_independent_cache(
    pixel_cache_path: Path,
    weighted_cache_path: Path,
    max_samples_per_frame: int = 160,
    sample_frame_stride: int = 4,
    seed: int = 7,
) -> tuple[pd.DataFrame, pd.DataFrame, str]:
    pixel_required = {
        "position_label",
        "time_index",
        "time_hours",
        "rfp_raw",
        "yfp_raw",
        "rfp_z",
        "yfp_z",
    }
    weighted_required = {
        "position_label",
        "time_index",
        "time_hours",
        "yfp_weighted_mean_rfp_percentile",
        "mask_pixel_count",
        "positive_yfp_weight_sum",
    }
    if pixel_cache_path.exists() and weighted_cache_path.exists():
        pixel_df = pd.read_csv(pixel_cache_path, sep="\t")
        weighted_df = pd.read_csv(weighted_cache_path, sep="\t")
        if pixel_required.issubset(pixel_df.columns) and weighted_required.issubset(weighted_df.columns):
            return pixel_df, weighted_df, "reused existing"

    pixel_df, weighted_df = threshold_independent_pixel_data(
        max_samples_per_frame=max_samples_per_frame,
        sample_frame_stride=sample_frame_stride,
        seed=seed,
    )
    pixel_df.to_csv(pixel_cache_path, sep="\t", index=False)
    weighted_df.to_csv(weighted_cache_path, sep="\t", index=False)
    return pixel_df, weighted_df, "created new"


def quadrant_percentages_from_raw(
    df: pd.DataFrame,
    rfp_cut: float,
    yfp_cut: float,
) -> dict[str, float]:
    if df.empty:
        return {"UL": float("nan"), "UR": float("nan"), "LL": float("nan"), "LR": float("nan")}
    total = float(len(df))
    x = df["rfp_raw"].to_numpy(dtype=float)
    y = df["yfp_raw"].to_numpy(dtype=float)
    return {
        "UL": float(np.sum((x < rfp_cut) & (y >= yfp_cut)) / total),
        "UR": float(np.sum((x >= rfp_cut) & (y >= yfp_cut)) / total),
        "LL": float(np.sum((x < rfp_cut) & (y < yfp_cut)) / total),
        "LR": float(np.sum((x >= rfp_cut) & (y < yfp_cut)) / total),
    }


def nearest_time_subset(summary_df: pd.DataFrame, target_times: list[float]) -> pd.DataFrame:
    rows = []
    available = summary_df["time_hours"].to_numpy(dtype=float)
    for target in target_times:
        idx = int(np.argmin(np.abs(available - float(target))))
        rows.append(summary_df.iloc[idx])
    subset = pd.DataFrame(rows).drop_duplicates(subset=["time_hours"]).sort_values("time_hours").reset_index(drop=True)
    return subset


def plot_with_gap_breaks(
    ax,
    time_hours: np.ndarray,
    values: np.ndarray,
    gap_threshold_hours: float,
    **plot_kwargs,
) -> None:
    time_arr = np.asarray(time_hours, dtype=float)
    value_arr = np.asarray(values, dtype=float)
    finite = np.isfinite(time_arr) & np.isfinite(value_arr)
    time_arr = time_arr[finite]
    value_arr = value_arr[finite]
    if time_arr.size == 0:
        return
    split_points = np.flatnonzero(np.diff(time_arr) > float(gap_threshold_hours)) + 1
    for time_seg, value_seg in zip(np.split(time_arr, split_points), np.split(value_arr, split_points)):
        if time_seg.size == 0:
            continue
        ax.plot(time_seg, value_seg, **plot_kwargs)


## Theme 1: YFP Appears Within An Existing RFP Context

This notebook stays at the level of spatial context and threshold definition.

The theme is:

- `YFP` is most convincing if it appears inside an already established `RFP` context
- threshold choice should be justified before the timing notebooks are interpreted


### Quantitative Containment Over Time

The main view below uses the current default pairing:

- `RFP 4σ`
- `YFP 3σ`

and includes only frames with at least `50` `YFP`-positive pixels, so tiny unstable `YFP` masks do not dominate the statistic.


In [ ]:
containment_cache_path = TABLE_DIR / "05b_yfp_in_rfp_containment_by_frame.tsv"
containment_df, containment_cache_status = load_or_build_containment_cache(
    containment_cache_path,
    yfp_sigma_threshold=DEFAULT_YFP_SIGMA,
    rfp_sigma_threshold=DEFAULT_RFP_SIGMA,
    min_yfp_pixels=50,
)

containment_summary_df = (
    containment_df.groupby("time_hours")
    .agg(
        n_positions=("containment_fraction", "size"),
        mean_containment=("containment_fraction", "mean"),
        fraction_positions_with_containment_ge_0p95=(
            "containment_fraction",
            lambda s: float(np.mean(np.asarray(s, dtype=float) >= 0.95)),
        ),
    )
    .reset_index()
)
containment_summary_path = TABLE_DIR / "05b_yfp_in_rfp_containment_summary.tsv"
display_time_df(containment_summary_df).to_csv(containment_summary_path, sep="\t", index=False)

prevalence_display_df = nearest_time_subset(
    containment_summary_df,
    [0, 4, 8, 12, 16, 24, 32, 40, 48, 56, 64, 68],
)

fig, ax = plt.subplots(1, 1, figsize=(5.6, 4.3), constrained_layout=True)
for position_label, position_df in containment_df.groupby("position_label", sort=True):
    position_df = position_df.sort_values("time_hours")
    plot_with_gap_breaks(
        ax,
        position_df["time_hours"],
        position_df["containment_fraction"],
        gap_threshold_hours=GAP_BREAK_THRESHOLD_HOURS,
        color="0.72",
        linewidth=0.8,
        alpha=0.42,
    )
ax.plot(
    containment_summary_df["time_hours"],
    containment_summary_df["mean_containment"],
    color="#4c78a8",
    linewidth=2.8,
    label="mean containment",
)
ax.set_ylim(0.0, 1.02)
ax.set_xlabel("Time (hours)")
set_display_time_axis(ax, "x")
ax.set_ylabel("Fraction of BMP4-YFP-positive pixels also FOXF1-RFP-positive")
ax.set_title("BMP4-YFP containment within FOXF1-RFP", fontsize=10.0)
ax.grid(alpha=0.18)
ax.legend(frameon=False, loc="lower left")

containment_reference_path = figure_path("05b_yfp_in_rfp_containment.png", candidate=False)
fig.suptitle("Quantitative BMP4-YFP-within-FOXF1-RFP containment", fontsize=11.6)
fig.savefig(containment_reference_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, ax = plt.subplots(1, 1, figsize=(5.4, 4.3), constrained_layout=True)
ax.plot(
    containment_summary_df["time_hours"],
    containment_summary_df["fraction_positions_with_containment_ge_0p95"],
    color="#d62728",
    linewidth=2.0,
    alpha=0.95,
)
ax.scatter(
    prevalence_display_df["time_hours"],
    prevalence_display_df["fraction_positions_with_containment_ge_0p95"],
    color="#d62728",
    s=28,
    zorder=3,
)
for idx, row in prevalence_display_df.iterrows():
    offset = 0.035 if (idx % 2 == 0) else -0.055
    ax.text(
        float(row["time_hours"]),
        float(row["fraction_positions_with_containment_ge_0p95"]) + offset,
        f"n={int(row['n_positions'])}",
        ha="center",
        va="bottom" if offset > 0 else "top",
        fontsize=7.4,
        color="0.25",
        bbox={"boxstyle": "round,pad=0.12", "fc": "white", "ec": "0.85", "alpha": 0.92},
    )
ax.set_ylim(0.0, 1.02)
ax.set_xlabel("Time (hours)")
set_display_time_axis(ax, "x")
ax.set_ylabel("Fraction of positions with containment ≥ 0.95")
ax.set_title("Strong-containment prevalence", fontsize=10.0)
ax.grid(alpha=0.18)

containment_prevalence_path = figure_path("05b_strong_containment_prevalence.png")
fig.suptitle("Supplementary containment prevalence summary", fontsize=11.6)
fig.savefig(containment_prevalence_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

containment_display_df = prevalence_display_df[
    [
        "time_hours",
        "n_positions",
        "mean_containment",
        "fraction_positions_with_containment_ge_0p95",
    ]
].copy()
display(Markdown(f"**Containment cache:** `{containment_cache_status}`"))
display(Markdown("**Containment summary at displayed timepoints**"))
display(display_time_df(containment_display_df).style.hide(axis="index"))
print("Wrote figure:", containment_reference_path)
print("Wrote figure:", containment_prevalence_path)
print("Wrote table:", containment_cache_path)
print("Wrote table:", containment_summary_path)


### YFP Threshold Sensitivity Of The Containment Readout

This holds `RFP` fixed at `4 sigma` and varies only the `YFP` threshold.

The point is not to find one perfect number immediately, but to see whether `2`, `3`, `4`, and `5 sigma` materially change the basic nested-domain picture.


In [ ]:
containment_threshold_path = TABLE_DIR / "05b_yfp_in_rfp_containment_yfp_threshold_sensitivity.tsv"
threshold_required_cols = {
    "position_label",
    "time_index",
    "time_hours",
    "rfp_sigma_threshold",
    "yfp_sigma_threshold",
    "rfp_positive_pixels",
    "yfp_positive_pixels",
    "intersection_pixels",
    "containment_fraction",
}
if containment_threshold_path.exists():
    cached = pd.read_csv(containment_threshold_path, sep="\t")
    if threshold_required_cols.issubset(cached.columns) and set(cached["rfp_sigma_threshold"].dropna().unique()) == {4}:
        containment_threshold_df = cached.loc[cached["yfp_positive_pixels"] >= 50].copy()
        threshold_cache_status = "reused existing"
    else:
        raise RuntimeError("Existing threshold-sensitivity cache has unexpected columns or thresholds.")
else:
    yfp_sigma_choices = [2, 3, 4, 5]
    containment_threshold_frames = []
    for yfp_sigma in yfp_sigma_choices:
        sigma_df = containment_by_time(
            yfp_sigma_threshold=yfp_sigma,
            rfp_sigma_threshold=4,
            min_yfp_pixels=50,
        ).copy()
        sigma_df["yfp_sigma_threshold"] = int(yfp_sigma)
        containment_threshold_frames.append(sigma_df)
    containment_threshold_df = pd.concat(containment_threshold_frames, ignore_index=True)
    containment_threshold_df.to_csv(containment_threshold_path, sep="\t", index=False)
    threshold_cache_status = "created new"

yfp_sigma_choices = [2, 3, 4, 5]
containment_threshold_summary_df = (
    containment_threshold_df.groupby(["yfp_sigma_threshold", "time_hours"])
    .agg(
        n_positions=("containment_fraction", "size"),
        mean_containment=("containment_fraction", "mean"),
        fraction_positions_with_containment_ge_0p95=(
            "containment_fraction",
            lambda s: float(np.mean(np.asarray(s, dtype=float) >= 0.95)),
        ),
    )
    .reset_index()
)
containment_threshold_summary_path = TABLE_DIR / "05b_yfp_in_rfp_containment_yfp_threshold_sensitivity_summary.tsv"
display_time_df(containment_threshold_summary_df).to_csv(containment_threshold_summary_path, sep="\t", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2), constrained_layout=True)
sigma_colors = {2: "#9ecae1", 3: "#4c78a8", 4: "#fdae6b", 5: "#d62728"}
for yfp_sigma in yfp_sigma_choices:
    subset = containment_threshold_summary_df.loc[
        containment_threshold_summary_df["yfp_sigma_threshold"] == yfp_sigma
    ].sort_values("time_hours")
    axes[0].plot(
        subset["time_hours"],
        subset["mean_containment"],
        color=sigma_colors[yfp_sigma],
        linewidth=2.4,
        label=f"BMP4-YFP {yfp_sigma}σ",
    )
    axes[1].plot(
        subset["time_hours"],
        subset["fraction_positions_with_containment_ge_0p95"],
        color=sigma_colors[yfp_sigma],
        linewidth=2.4,
        label=f"BMP4-YFP {yfp_sigma}σ",
    )

for ax in axes:
    ax.set_ylim(0.0, 1.02)
    ax.grid(alpha=0.18)
    ax.legend(frameon=False, fontsize=8, loc="lower left")
axes[0].set_xlabel("Time (hours)")
set_display_time_axis(axes[0], "x")
axes[0].set_ylabel("Mean containment")
axes[0].set_title("Containment mean", fontsize=10.0)
axes[1].set_xlabel("Time (hours)")
set_display_time_axis(axes[1], "x")
axes[1].set_ylabel("Fraction of positions with containment ≥ 0.95")
axes[1].set_title("Strong-containment prevalence", fontsize=10.0)

containment_threshold_fig_path = figure_path("05b_yfp_in_rfp_containment_yfp_threshold_sensitivity.png")
fig.suptitle("Containment sensitivity to BMP4-YFP threshold (FOXF1-RFP fixed at 4σ)", fontsize=11.6)
fig.savefig(containment_threshold_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

display(Markdown(f"**Threshold-sensitivity cache:** `{threshold_cache_status}`"))
threshold_display_times = [16.0, 40.0, 56.0]
threshold_display_df = containment_threshold_summary_df.loc[
    containment_threshold_summary_df["time_hours"].isin(threshold_display_times)
].copy()
threshold_display_df = threshold_display_df[
    [
        "yfp_sigma_threshold",
        "time_hours",
        "n_positions",
        "mean_containment",
        "fraction_positions_with_containment_ge_0p95",
    ]
].sort_values(["time_hours", "yfp_sigma_threshold"])
display(Markdown("**Threshold-sensitivity summary at representative times**"))
display(display_time_df(threshold_display_df).style.hide(axis="index"))
print("Wrote figure:", containment_threshold_fig_path)
print("Wrote table:", containment_threshold_path)
print("Wrote table:", containment_threshold_summary_path)


### Threshold-Independent Supporting Views

The thresholded containment analysis is the main story for this notebook.

The views below are meant only as threshold-independent support:

- pooled within-cyst `FOXF1-RFP`/`BMP4-YFP` pixel density
- the same density split into the four time windows chosen in `05c`
- one compact asymmetric summary statistic, reported per position rather than as a timecourse

Selection manifests written in this section:

- sampled within-cyst pixel cache: `results/tables/05b_threshold_independent_pixel_joint_sample.tsv`
- time-window split used for the 2x2 density montage: `results/tables/05b_threshold_independent_pixel_density_time_window_selection.tsv`


In [ ]:
pixel_joint_path = TABLE_DIR / "05b_threshold_independent_pixel_joint_sample.tsv"
weighted_percentile_path = TABLE_DIR / "05b_threshold_independent_weighted_rfp_percentile.tsv"
pixel_joint_df, weighted_percentile_df, threshold_independent_cache_status = load_or_build_threshold_independent_cache(
    pixel_joint_path,
    weighted_percentile_path,
    max_samples_per_frame=160,
    sample_frame_stride=4,
    seed=7,
)

rfp_floor, rfp_transform = build_shifted_log_transform(pixel_joint_df["rfp_raw"].to_numpy(dtype=float))
yfp_floor, yfp_transform = build_shifted_log_transform(pixel_joint_df["yfp_raw"].to_numpy(dtype=float))
pixel_joint_df["rfp_display"] = rfp_transform(pixel_joint_df["rfp_raw"].to_numpy(dtype=float))
pixel_joint_df["yfp_display"] = yfp_transform(pixel_joint_df["yfp_raw"].to_numpy(dtype=float))

rfp_threshold_display = float(rfp_transform(default_rfp_threshold_value))
yfp_threshold_display = float(yfp_transform(default_yfp_threshold_value))

x_limits = (
    float(np.quantile(pixel_joint_df["rfp_display"], 0.001)),
    float(np.quantile(pixel_joint_df["rfp_display"], 0.999)),
)
y_limits = (
    float(np.quantile(pixel_joint_df["yfp_display"], 0.001)),
    float(np.quantile(pixel_joint_df["yfp_display"], 0.999)),
)

overall_quadrants = quadrant_percentages_from_raw(
    pixel_joint_df,
    rfp_cut=default_rfp_threshold_value,
    yfp_cut=default_yfp_threshold_value,
)

hist_range = [x_limits, y_limits]
hist2d, x_edges, y_edges = np.histogram2d(
    pixel_joint_df["rfp_display"].to_numpy(dtype=float),
    pixel_joint_df["yfp_display"].to_numpy(dtype=float),
    bins=120,
    range=hist_range,
)
hist2d_log = np.log10(hist2d + 1.0)
hist2d_smooth = gaussian_filter(hist2d.astype(float), sigma=1.25)
hist2d_smooth_log = np.log10(hist2d_smooth + 1.0)
hist2d_log_masked = np.ma.masked_where(hist2d <= 0, hist2d_log)
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
positive_smooth = hist2d_smooth_log[hist2d_smooth_log > 0]
contour_levels = np.quantile(positive_smooth, [0.55, 0.75, 0.90, 0.97]) if positive_smooth.size else np.array([0.25, 0.5, 0.75, 1.0])
density_cmap = plt.cm.magma.copy()
density_cmap.set_bad("black")

fig, ax = plt.subplots(figsize=(6.0, 4.9), constrained_layout=True)
mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    hist2d_log_masked.T,
    cmap=density_cmap,
    shading="auto",
    rasterized=True,
)
ax.contour(
    x_centers,
    y_centers,
    hist2d_smooth_log.T,
    levels=np.unique(contour_levels),
    colors="white",
    linewidths=0.9,
    alpha=0.9,
)
ax.axvline(rfp_threshold_display, color="white", linestyle="--", linewidth=1.2)
ax.axhline(yfp_threshold_display, color="white", linestyle="--", linewidth=1.2)
ax.set_xlim(*x_limits)
ax.set_ylim(*y_limits)
ax.set_xlabel("FOXF1-RFP shifted-log corrected intensity (display units)")
ax.set_ylabel("BMP4-YFP shifted-log corrected intensity (display units)")
ax.set_title("Pooled within-cyst pixel density", fontsize=10.2)

quadrant_locations = {
    "UL": (0.03, 0.97),
    "UR": (0.97, 0.97),
    "LL": (0.03, 0.03),
    "LR": (0.97, 0.03),
}
for key, (x_pos, y_pos) in quadrant_locations.items():
    ax.text(
        x_pos,
        y_pos,
        f"{key}: {overall_quadrants[key] * 100:.1f}%",
        transform=ax.transAxes,
        ha="left" if "L" in key else "right",
        va="top" if "U" in key else "bottom",
        fontsize=8.2,
        color="white",
        bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=2.0),
    )
fig.colorbar(mesh, ax=ax, label="Binned pixel count (log10 count + 1)")
pooled_density_reference_path = figure_path("05b_threshold_independent_pixel_density.png")
fig.suptitle("Threshold-independent within-cyst FOXF1-RFP/BMP4-YFP pixel density", fontsize=11.6)
fig.savefig(pooled_density_reference_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", pooled_density_reference_path)

time_bin_edges = [0.0, 6.0, 12.0, 30.0, 68.25]
time_bin_labels = [
    format_display_hour_range(0.0, 6.0),
    format_display_hour_range(6.0, 12.0),
    format_display_hour_range(12.0, 30.0),
    format_display_hour_range(30.0, 68.25, decimals=1),
]
time_window_selection_df = pd.DataFrame(
    [
        {
            "window_rank": idx + 1,
            "time_window_label": label,
            "start_time_hours": float(start_time),
            "end_time_hours": float(end_time),
            "n_sampled_pixels": int(
                (
                    (pixel_joint_df["time_hours"] >= float(start_time))
                    & (pixel_joint_df["time_hours"] < float(end_time))
                ).sum()
            ),
            "n_positions": int(
                pixel_joint_df.loc[
                    (pixel_joint_df["time_hours"] >= float(start_time))
                    & (pixel_joint_df["time_hours"] < float(end_time)),
                    "position_label",
                ].nunique()
            ),
            "n_position_time_frames": int(
                pixel_joint_df.loc[
                    (pixel_joint_df["time_hours"] >= float(start_time))
                    & (pixel_joint_df["time_hours"] < float(end_time)),
                    ["position_label", "time_index"],
                ].drop_duplicates().shape[0]
            ),
        }
        for idx, (label, start_time, end_time) in enumerate(
            zip(time_bin_labels, time_bin_edges[:-1], time_bin_edges[1:])
        )
    ]
)
time_window_selection_path = TABLE_DIR / "05b_threshold_independent_pixel_density_time_window_selection.tsv"
display_time_df(time_window_selection_df).to_csv(time_window_selection_path, sep="\t", index=False)
pixel_joint_df["time_window_label"] = pd.cut(
    pixel_joint_df["time_hours"],
    bins=time_bin_edges,
    labels=time_bin_labels,
    include_lowest=True,
    right=False,
)

fig, axes = plt.subplots(2, 2, figsize=(10.2, 8.8), constrained_layout=True)
last_mesh = None
for ax, label in zip(axes.ravel(), time_bin_labels):
    subset = pixel_joint_df.loc[pixel_joint_df["time_window_label"] == label].copy()
    if subset.empty:
        ax.axis("off")
        continue
    subset_hist2d, subset_x_edges, subset_y_edges = np.histogram2d(
        subset["rfp_display"].to_numpy(dtype=float),
        subset["yfp_display"].to_numpy(dtype=float),
        bins=70,
        range=hist_range,
    )
    subset_hist2d_log = np.log10(subset_hist2d + 1.0)
    subset_hist2d_smooth = gaussian_filter(subset_hist2d.astype(float), sigma=1.0)
    subset_hist2d_smooth_log = np.log10(subset_hist2d_smooth + 1.0)
    subset_hist2d_log_masked = np.ma.masked_where(subset_hist2d <= 0, subset_hist2d_log)
    subset_x_centers = 0.5 * (subset_x_edges[:-1] + subset_x_edges[1:])
    subset_y_centers = 0.5 * (subset_y_edges[:-1] + subset_y_edges[1:])
    subset_positive_smooth = subset_hist2d_smooth_log[subset_hist2d_smooth_log > 0]
    subset_levels = (
        np.quantile(subset_positive_smooth, [0.60, 0.78, 0.90, 0.97])
        if subset_positive_smooth.size
        else np.array([0.25, 0.5, 0.75, 1.0])
    )
    last_mesh = ax.pcolormesh(
        subset_x_edges,
        subset_y_edges,
        subset_hist2d_log_masked.T,
        cmap=density_cmap,
        shading="auto",
        rasterized=True,
    )
    ax.contour(
        subset_x_centers,
        subset_y_centers,
        subset_hist2d_smooth_log.T,
        levels=np.unique(subset_levels),
        colors="white",
        linewidths=0.8,
        alpha=0.85,
    )
    quadrants = quadrant_percentages_from_raw(
        subset,
        rfp_cut=default_rfp_threshold_value,
        yfp_cut=default_yfp_threshold_value,
    )
    ax.axvline(rfp_threshold_display, color="white", linestyle="--", linewidth=1.0)
    ax.axhline(yfp_threshold_display, color="white", linestyle="--", linewidth=1.0)
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_title(label, fontsize=9.6)
    ax.set_xlabel("FOXF1-RFP display")
    ax.set_ylabel("BMP4-YFP display")
    for key, (x_pos, y_pos) in quadrant_locations.items():
        ax.text(
            x_pos,
            y_pos,
            f"{quadrants[key] * 100:.1f}%",
            transform=ax.transAxes,
            ha="left" if "L" in key else "right",
            va="top" if "U" in key else "bottom",
            fontsize=7.6,
            color="white",
            bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=1.5),
        )
if last_mesh is not None:
    fig.colorbar(last_mesh, ax=axes.ravel().tolist(), shrink=0.84, label="Binned pixel count (log10 count + 1)")
facs_time_path = figure_path("05b_threshold_independent_pixel_density_time_windows.png")
fig.suptitle("Threshold-independent within-cyst pixel density by time window", fontsize=11.6)
fig.savefig(facs_time_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", facs_time_path)

per_position_weighted_df = (
    weighted_percentile_df.groupby("position_label")
    .agg(
        n_frames=("yfp_weighted_mean_rfp_percentile", lambda s: int(np.isfinite(np.asarray(s, dtype=float)).sum())),
        mean_weighted_rfp_percentile=("yfp_weighted_mean_rfp_percentile", "mean"),
        median_weighted_rfp_percentile=("yfp_weighted_mean_rfp_percentile", "median"),
    )
    .reset_index()
)
valid_weighted = per_position_weighted_df["mean_weighted_rfp_percentile"].to_numpy(dtype=float)
valid_weighted = valid_weighted[np.isfinite(valid_weighted)]
if valid_weighted.size:
    wilcoxon_result = stats.wilcoxon(valid_weighted - 0.5, alternative="greater")
    wilcoxon_pvalue = float(wilcoxon_result.pvalue)
else:
    wilcoxon_pvalue = float("nan")

weighted_summary_df = pd.DataFrame(
    [
        {
            "n_positions": int(valid_weighted.size),
            "mean_position_statistic": float(np.nanmean(valid_weighted)) if valid_weighted.size else float("nan"),
            "median_position_statistic": float(np.nanmedian(valid_weighted)) if valid_weighted.size else float("nan"),
            "fraction_positions_above_0p5": float(np.mean(valid_weighted > 0.5)) if valid_weighted.size else float("nan"),
            "wilcoxon_pvalue_vs_0p5_greater": wilcoxon_pvalue,
        }
    ]
)
per_position_weighted_path = TABLE_DIR / "05b_threshold_independent_weighted_rfp_percentile_by_position.tsv"
weighted_summary_path = TABLE_DIR / "05b_threshold_independent_weighted_rfp_percentile_summary.tsv"
per_position_weighted_df.to_csv(per_position_weighted_path, sep="\t", index=False)
weighted_summary_df.to_csv(weighted_summary_path, sep="\t", index=False)

weighted_position_display_df = (
    pd.concat(
        [
            per_position_weighted_df.sort_values("mean_weighted_rfp_percentile").head(3),
            per_position_weighted_df.sort_values("mean_weighted_rfp_percentile").tail(3),
        ],
        ignore_index=True,
    )
    .drop_duplicates(subset=["position_label"])
    .sort_values("mean_weighted_rfp_percentile")
    .reset_index(drop=True)
)

display(Markdown(f"**Threshold-independent sample cache:** `{threshold_independent_cache_status}`"))
display(Markdown(f"**Time-window selection manifest:** `{time_window_selection_path.name}`"))
display(Markdown("**Threshold-independent summary statistic**"))
display(weighted_summary_df.style.hide(axis="index"))
display(Markdown("**Example per-position values (lowest and highest)**"))
display(weighted_position_display_df.style.hide(axis="index"))
print("Wrote table:", pixel_joint_path)
print("Wrote table:", time_window_selection_path)
print("Wrote table:", weighted_percentile_path)
print("Wrote table:", per_position_weighted_path)
print("Wrote table:", weighted_summary_path)


### Containment Outlier Review

These are not exclusion calls.

They are simply the positions whose containment stays unusually low over a sustained core window (`8-56 h`) under the current rule:

- at least `20` contributing core-window frames
- mean containment below `0.95`

This block is mainly for inspection. Low-area `YFP` masks near the minimum-size filter can look more outlying here than they really are biologically.

The exact low-containment representative frame chosen for each flagged position is written to:

- `results/tables/05b_containment_outlier_representative_frames.tsv`


In [ ]:
containment_core_df = containment_df.loc[
    (containment_df["time_hours"] >= 8.0)
    & (containment_df["time_hours"] <= 56.0)
].copy()
containment_outlier_summary_df = (
    containment_core_df.groupby("position_label")
    .agg(
        n_core_frames=("containment_fraction", "size"),
        mean_core_containment=("containment_fraction", "mean"),
        median_core_containment=("containment_fraction", "median"),
        min_core_containment=("containment_fraction", "min"),
        min_yfp_positive_pixels=("yfp_positive_pixels", "min"),
        median_yfp_positive_pixels=("yfp_positive_pixels", "median"),
    )
    .reset_index()
)
containment_outlier_summary_df = containment_outlier_summary_df.loc[
    (containment_outlier_summary_df["n_core_frames"] >= 20)
    & (containment_outlier_summary_df["mean_core_containment"] < 0.95)
].sort_values("mean_core_containment")
containment_outlier_summary_path = TABLE_DIR / "05b_containment_outlier_summary.tsv"
display_time_df(containment_outlier_summary_df).to_csv(containment_outlier_summary_path, sep="\t", index=False)

display(Markdown("**These are the only positions currently flagged by this outlier rule.**"))
display(display_time_df(containment_outlier_summary_df).style.hide(axis="index"))

if not containment_outlier_summary_df.empty:
    display_rows = []
    for row in containment_outlier_summary_df.itertuples(index=False):
        position_label = str(row.position_label)
        position_frames = containment_core_df.loc[
            containment_core_df["position_label"] == position_label
        ].sort_values("containment_fraction")
        representative = position_frames.iloc[0]
        display_rows.append(
            {
                "position_label": position_label,
                "time_index": int(representative["time_index"]),
                "time_hours": float(representative["time_hours"]),
                "containment_fraction": float(representative["containment_fraction"]),
            }
        )
    display_df = pd.DataFrame(display_rows)
    containment_outlier_representative_path = TABLE_DIR / "05b_containment_outlier_representative_frames.tsv"
    display_time_df(display_df).to_csv(containment_outlier_representative_path, sep="\t", index=False)
    display(Markdown(f"**Representative low-containment frames:** `{containment_outlier_representative_path.name}`"))

    rfp_arrays = []
    yfp_arrays = []
    for row in display_df.itertuples(index=False):
        stack = load_corrected_stack(str(row.position_label))
        mask = tiff.imread(organoid_mask_path(str(row.position_label), int(row.time_index))) > 0
        rfp_arrays.append(stack[int(row.time_index), 1][mask].astype(float))
        yfp_arrays.append(stack[int(row.time_index), 2][mask].astype(float))
    rfp_vmin = float(np.quantile(np.concatenate(rfp_arrays), 0.01))
    rfp_vmax = float(np.quantile(np.concatenate(rfp_arrays), 0.995))
    yfp_vmin = float(np.quantile(np.concatenate(yfp_arrays), 0.01))
    yfp_vmax = float(np.quantile(np.concatenate(yfp_arrays), 0.995))

    fig, axes = plt.subplots(len(display_df), 2, figsize=(8.8, 3.6 * len(display_df)), constrained_layout=True, squeeze=False)
    for row_idx, row in enumerate(display_df.itertuples(index=False)):
        position_label = str(row.position_label)
        time_index = int(row.time_index)
        stack = load_corrected_stack(position_label)
        mask = tiff.imread(organoid_mask_path(position_label, time_index)) > 0
        rfp_image = stack[time_index, 1].astype(float)
        yfp_image = stack[time_index, 2].astype(float)
        rfp_mask = positive_mask_from_corrected(rfp_image, mask, default_rfp_threshold_value)
        yfp_mask = positive_mask_from_corrected(yfp_image, mask, default_yfp_threshold_value)

        for ax, image, reporter, vmin, vmax in [
            (axes[row_idx, 0], rfp_image, "RFP", rfp_vmin, rfp_vmax),
            (axes[row_idx, 1], yfp_image, "YFP", yfp_vmin, yfp_vmax),
        ]:
            ax.imshow(image, cmap="magma", vmin=vmin, vmax=vmax)
            for contour in find_contours(mask.astype(float), 0.5):
                ax.plot(contour[:, 1], contour[:, 0], color="white", linewidth=1.0)
            for contour in find_contours(rfp_mask.astype(float), 0.5):
                ax.plot(contour[:, 1], contour[:, 0], color="#ff2d2d", linewidth=1.2)
            for contour in find_contours(yfp_mask.astype(float), 0.5):
                ax.plot(contour[:, 1], contour[:, 0], color="#f0c000", linewidth=1.2)
            ax.set_xticks([])
            ax.set_yticks([])
            if row_idx == 0:
                title_sigma = "4σ" if reporter == "RFP" else "3σ"
                ax.set_title(f"{reporter_display(reporter)} corrected image with {title_sigma} boundaries", fontsize=10.0)
        axes[row_idx, 0].set_ylabel(
            f"{position_label}\nt={format_display_hours(row.time_hours, 1)}\ncontainment={row.containment_fraction:.3f}",
            fontsize=9.2,
        )

    containment_outlier_fig_path = figure_path("05b_containment_outlier_review.png")
    fig.suptitle("Containment outlier review at representative low-containment frames", fontsize=11.6)
    fig.savefig(containment_outlier_fig_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", containment_outlier_fig_path)
    print("Wrote table:", containment_outlier_representative_path)
print("Wrote table:", containment_outlier_summary_path)


### Representative Domain Progression

The montage below uses the current shortlist of visually strong candidate cysts from `05aa`.

Display convention:

- white outline: organoid mask
- red fill/outline: `FOXF1-RFP`-positive region
- gold fill/outline: `BMP4-YFP`-positive region

The exact position/time grid used in this montage is written to:

- `results/tables/05b_representative_domain_progression_selection.tsv`


In [ ]:
representative_columns = [
    ("48 h", 0.0),
    ("72 h", 24.0),
    ("96 h", 48.0),
    ("final", None),
]
fig, axes = plt.subplots(
    len(POSITION_REVIEW_SET),
    len(representative_columns),
    figsize=(10.8, 13.2),
    constrained_layout=True,
)
if len(POSITION_REVIEW_SET) == 1:
    axes = np.asarray([axes])
representative_selection_rows = []

for row_idx, position_label in enumerate(POSITION_REVIEW_SET):
    stack = load_corrected_stack(position_label)
    phase_stack = stack[:, 0].astype(float)
    rfp_stack = stack[:, 1].astype(float)
    yfp_stack = stack[:, 2].astype(float)

    final_hours = float(final_time_hours_by_position.get(position_label, (stack.shape[0] - 1) * 0.25))

    for col_idx, (column_label, display_h) in enumerate(representative_columns):
        ax = axes[row_idx, col_idx]
        time_hours = final_hours if display_h is None else float(display_h)
        time_index = int(round(time_hours / 0.25))
        mask_path = organoid_mask_path(position_label, time_index)
        representative_selection_rows.append(
            {
                "position_label": position_label,
                "column_label": column_label,
                "requested_time_hours": float(final_hours if display_h is None else display_h),
                "selected_time_hours": float(time_hours),
                "selected_time_index": int(time_index),
                "is_final_frame": bool(display_h is None),
                "mask_exists": bool(mask_path.exists()),
            }
        )
        if not mask_path.exists():
            ax.axis("off")
            continue

        organoid_mask = tiff.imread(mask_path) > 0
        phase_image = phase_stack[time_index]
        lo, hi = np.percentile(phase_image, [1, 99])
        phase_image = exposure.rescale_intensity(phase_image, in_range=(lo, hi))
        rfp_mask = positive_mask_from_corrected(rfp_stack[time_index], organoid_mask, default_rfp_threshold_value)
        yfp_mask = positive_mask_from_corrected(yfp_stack[time_index], organoid_mask, default_yfp_threshold_value)

        ax.imshow(phase_image, cmap="gray", interpolation="nearest")

        rfp_overlay = np.zeros((*organoid_mask.shape, 4), dtype=float)
        rfp_overlay[..., 0] = 1.0
        rfp_overlay[..., 3] = rfp_mask.astype(float) * 0.22
        ax.imshow(rfp_overlay, interpolation="nearest")

        yfp_overlay = np.zeros((*organoid_mask.shape, 4), dtype=float)
        yfp_overlay[..., 0] = 0.95
        yfp_overlay[..., 1] = 0.75
        yfp_overlay[..., 3] = yfp_mask.astype(float) * 0.55
        ax.imshow(yfp_overlay, interpolation="nearest")

        for contour in find_contours(organoid_mask.astype(float), 0.5):
            ax.plot(contour[:, 1], contour[:, 0], color="white", linewidth=1.1)
        for contour in find_contours(rfp_mask.astype(float), 0.5):
            ax.plot(contour[:, 1], contour[:, 0], color="#ff2d2d", linewidth=1.2)
        for contour in find_contours(yfp_mask.astype(float), 0.5):
            ax.plot(contour[:, 1], contour[:, 0], color="#f0c000", linewidth=1.4)

        ax.set_xticks([])
        ax.set_yticks([])
        if row_idx == 0:
            ax.set_title(column_label, fontsize=10.5)
        if col_idx == 0:
            ax.set_ylabel(position_label, fontsize=9.8)

representative_selection_df = pd.DataFrame(representative_selection_rows)
representative_selection_path = TABLE_DIR / "05b_representative_domain_progression_selection.tsv"
display_time_df(representative_selection_df).to_csv(representative_selection_path, sep="\t", index=False)
display(Markdown(f"**Representative domain-progression selection manifest:** `{representative_selection_path.name}`"))
representative_path = figure_path("05b_representative_domain_progression.png")
fig.suptitle("Representative domain progression over time", fontsize=11.6)
fig.savefig(representative_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", representative_path)
print("Wrote table:", representative_selection_path)


### Main Figure Candidates

The themed sections above generate the full reference set first.

The candidate figures are assembled here at the end so we can evaluate the strongest Theme 1 views together without mixing them into the main analysis flow.


In [ ]:
candidate_1_source = figure_path("05b_yfp_in_rfp_containment.png")
candidate_1_path = figure_path("yfp_in_rfp_containment.png", candidate=True)
candidate_2_source = figure_path("05b_threshold_independent_pixel_density.png")
candidate_2_path = figure_path("threshold_independent_pixel_density.png", candidate=True)

for source_path, target_path in [
    (candidate_1_source, candidate_1_path),
    (candidate_2_source, candidate_2_path),
]:
    for suffix in [".png", ".pdf", ".svg"]:
        source_variant = source_path.with_suffix(suffix)
        if source_variant.exists():
            shutil.copy2(source_variant, target_path.with_suffix(suffix))

display(Markdown("#### Candidate 1: BMP4-YFP containment within FOXF1-RFP"))
display(Image(filename=str(candidate_1_path)))

display(Markdown("#### Candidate 2: Threshold-independent within-cyst FOXF1-RFP/BMP4-YFP pixel density"))
display(Image(filename=str(candidate_2_path)))

print("Wrote figure:", candidate_1_path)
print("Wrote figure:", candidate_2_path)
